# Projekt · Woche 9 · Eval für euren Assistenten
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Ab dieser Woche messt ihr euren Assistenten bei jeder Änderung. Dieses Notebook enthält:
1. euren Assistenten aus Woche 8 (Daten, Wissensbasis, Tools, Schleife). **Fügt euer eigenes Tool aus Woche 8 an der markierten Stelle ein.**
2. ein Testset-Gerüst mit Beispielfällen für euren Track. **Ergänzt es auf mindestens 20 Fälle**, je mindestens 5 normal, Rand, negativ. Eure Testfragen aus dem Steckbrief sind der Startpunkt.
3. das Eval-Harness: Code-Checks, LLM-Judge, Wilson-Intervall
4. eine Ergebnis-Historie: Jeder Lauf wird mit Datum und Notiz gespeichert

**Ziel bis Woche 10:** Baseline mit Konfidenzintervall im Repo.

In [ ]:
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

In [ ]:
import os, glob, zipfile, csv, json, re, datetime
import numpy as np

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))

TRACK = "B" if os.path.exists(f"{ORDNER}/wartungshandbuch.md") else "A"
HEUTE = datetime.date.today()
print("Track", TRACK, "| heute:", HEUTE)

def lies_csv(name):
    with open(f"{ORDNER}/{name}", encoding="utf-8") as f:
        return list(csv.DictReader(f))

## 1 · Euer Assistent aus Woche 8

In [ ]:
from sentence_transformers import SentenceTransformer

def chunk_struktur(markdown, dokument):
    chunks, pfad, zeilen, titel = [], {}, [], dokument
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = dokument + " › " + " › ".join(pfad[e] for e in sorted(pfad))
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = []
for d in sorted(glob.glob(f"{ORDNER}/**/*.md", recursive=True)):
    if not os.path.basename(d).lower().startswith("readme"):
        CHUNKS += chunk_struktur(open(d, encoding="utf-8").read(), os.path.basename(d))
modell = SentenceTransformer("intfloat/multilingual-e5-small")
E = modell.encode(["passage: " + c["titel"] + "\n" + c["text"] for c in CHUNKS], normalize_embeddings=True)
print(len(CHUNKS), "Chunks in der Wissensbasis")

def search_knowledge_base(query, k=4):
    q = modell.encode(["query: " + query], normalize_embeddings=True)[0]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in np.argsort(E @ q)[::-1][:k]]

THEMEN = "Wartungshandbuch der Nordwerk GmbH (Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes)" if TRACK == "B" \
         else "Satzung und Sitzungsprotokolle der Datenfabrik Karlsruhe e.V. (Regeln, Beschlüsse, Budgets, Zuständigkeiten)"
TOOL_SUCHE = {
    "name": "search_knowledge_base",
    "description": f"Durchsucht {THEMEN} und liefert die passendsten Abschnitte mit Quelle. Nutze es für Fragen zu Regeln, Abläufen und dokumentiertem Wissen.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Suchanfrage in natürlicher Sprache."}}, "required": ["query"]},
}

In [ ]:
FUNKTIONEN = {"search_knowledge_base": search_knowledge_base}
TOOLS = [TOOL_SUCHE]
SCHREIBENDE_TOOLS = set()

if TRACK == "B":
    MASCHINEN = {m["id"]: m for m in lies_csv("maschinen.csv")}
    TICKETS = lies_csv("stoerungen.csv")
    TICKET_LOG = []

    def _pruefe(machine_id):
        if machine_id not in MASCHINEN:
            raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: {', '.join(MASCHINEN)}.")

    def get_machine_status(machine_id):
        _pruefe(machine_id)
        m = MASCHINEN[machine_id]
        offen = sum(t["maschine"] == machine_id and t["status"] == "offen" for t in TICKETS)
        return {**m, "offene_tickets": offen}

    def list_tickets(machine_id=None, status="offen"):
        if machine_id: _pruefe(machine_id)
        return [t for t in TICKETS + TICKET_LOG if (machine_id is None or t["maschine"] == machine_id) and (status == "alle" or t["status"] == status)]

    def create_ticket(machine_id, description, priority):
        _pruefe(machine_id)
        ticket = {"ticket": f"T-{1001 + len(TICKET_LOG)}", "datum": HEUTE.isoformat(), "maschine": machine_id,
                  "prioritaet": priority, "status": "offen", "beschreibung": description}
        TICKET_LOG.append(ticket)
        return {"ticket": ticket["ticket"], "status": "angelegt"}

    ID_TEXT = "Maschinen-ID im Format M-01 bis M-08."
    FUNKTIONEN |= {"get_machine_status": get_machine_status, "list_tickets": list_tickets, "create_ticket": create_ticket}
    TOOLS += [
        {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, Standort, offene Tickets.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}}, "required": ["machine_id"], "additionalProperties": False}},
        {"name": "list_tickets", "description": "Listet Störungstickets, standardmäßig nur offene. Optional für eine Maschine; status 'alle' zeigt auch geschlossene.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "status": {"type": "string", "enum": ["offen", "geschlossen", "alle"]}}}},
        {"name": "create_ticket", "description": "Legt ein neues Störungsticket an. Nur verwenden, wenn der Nutzer ausdrücklich ein Ticket möchte. Prioritäten: hoch = Maschine steht oder Sicherheitsrisiko, mittel = eingeschränkt, niedrig = kein Einfluss.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "description": {"type": "string"}, "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}},
                          "required": ["machine_id", "description", "priority"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"create_ticket"}

else:
    TERMINE = lies_csv("termine.csv")
    MITGLIEDER = {m["id"]: m for m in lies_csv("mitglieder.csv")}

    def _semester_index(text):
        jahr = int(re.search(r"\d{4}", text).group())
        return jahr * 2 + (1 if text.startswith("WS") else 0)

    def list_events(ab_datum=None, art=None):
        ab = ab_datum or HEUTE.isoformat()
        return [t for t in TERMINE if t["datum"] >= ab and (art is None or t["art"].lower() == art.lower())][:8]

    def get_member_engagement(member_id):
        if member_id not in MITGLIEDER:
            raise ValueError(f"Unbekannte Mitglieds-ID '{member_id}'. Format: D-001.")
        m = MITGLIEDER[member_id]
        aktuell = _semester_index(f"WS {HEUTE.year}" if HEUTE.month >= 10 else f"SoSe {HEUTE.year}")
        # Bewusst nur die Daten, die für eine Bescheinigung nötig sind: keine E-Mail, kein Studiengang
        return {"id": m["id"], "name": f"{m['vorname']} {m['nachname']}".strip(), "rolle": m["rolle"], "ressort": m["ressort"],
                "eintritt": m["eintritt"], "semester_seit_eintritt": aktuell - _semester_index(m["eintritt"]) + 1,
                "engagement_stunden": int(m["engagement_stunden"]), "status": m["status"]}

    def draft_certificate(member_id, taetigkeiten):
        d = get_member_engagement(member_id)
        os.makedirs("bescheinigungen", exist_ok=True)
        pfad = f"bescheinigungen/{member_id}_entwurf.md"
        with open(pfad, "w", encoding="utf-8") as f:
            f.write(f"# Engagementbescheinigung (Entwurf)\n\n{d['name']} ist seit dem {d['eintritt']} Mitglied der Datenfabrik Karlsruhe e.V. "
                    f"und hat sich in dieser Zeit mit rund {d['engagement_stunden']} Stunden engagiert, zuletzt als {d['rolle']}.\n\n"
                    f"Wesentliche Tätigkeiten: {taetigkeiten}\n\nKarlsruhe, {HEUTE:%d.%m.%Y}\n\n(Unterschrift zweier Vorstandsmitglieder)\n")
        return {"entwurf": pfad, "status": "angelegt"}

    FUNKTIONEN |= {"list_events": list_events, "get_member_engagement": get_member_engagement, "draft_certificate": draft_certificate}
    TOOLS += [
        {"name": "list_events", "description": "Listet kommende Termine des Vereins ab einem Datum (Standard: heute), optional nach Art (Sitzung, Workshop, Event, Exkursion, Vortrag, Mitgliederversammlung).",
         "parameters": {"type": "object", "properties": {"ab_datum": {"type": "string", "description": "Datum im Format JJJJ-MM-TT."}, "art": {"type": "string"}}}},
        {"name": "get_member_engagement", "description": "Liefert die für eine Engagementbescheinigung nötigen Daten eines Mitglieds: Name, Rolle, Eintritt, Semester, Engagementstunden, Status. Keine Kontaktdaten.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Mitglieds-ID im Format D-001."}}, "required": ["member_id"], "additionalProperties": False}},
        {"name": "draft_certificate", "description": "Erstellt einen Entwurf einer Engagementbescheinigung als Datei. Vorher mit get_member_engagement und den Regeln aus der Satzung (§ 13) prüfen, ob die Voraussetzungen erfüllt sind.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "taetigkeiten": {"type": "string", "description": "Wesentliche Tätigkeiten in einem Satz."}},
                          "required": ["member_id", "taetigkeiten"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"draft_certificate"}

print("Tools:", [t["name"] for t in TOOLS], "| schreibend:", SCHREIBENDE_TOOLS)

In [ ]:
# ▶ Fügt hier euer eigenes Tool aus Woche 8 ein (Funktion, FUNKTIONEN-Eintrag, TOOLS-Eintrag, ggf. SCHREIBENDE_TOOLS)

In [ ]:
import math, time
from openai import OpenAI
from typing import Literal
from pydantic import BaseModel, Field
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL, JUDGE_MODEL = "kit.mistral-small-4-119b-a8b", "kit.deepseek-v4-flash"   # System und Judge aus verschiedenen Modellfamilien

SYSTEM = (f"Du bist der Assistent unseres Projekts (Track {TRACK}). Heute ist der {HEUTE:%d.%m.%Y}. "
          "Nutze die Tools, statt Fakten zu raten. Fehlt eine nötige Angabe, frage nach. "
          "Steht die Antwort in keinem Tool-Ergebnis, sage das klar. "
          "Nenne bei Informationen aus der Wissensbasis die Quelle. Gib keine Kontaktdaten von Personen heraus. Antworte knapp auf Deutsch.")

def fuehre_tool_aus(name, eingabe):
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def frage_assistent(frage, system=None, eval_modus=False, max_runden=6):
    """Gibt (Antworttext, Verlauf, Tokens) zurück. Im Eval-Modus werden schreibende Tools nie ausgeführt."""
    nachrichten, tokens = [{"role": "system", "content": system or SYSTEM}, {"role": "user", "content": frage}], {"input": 0, "output": 0}
    for _ in range(max_runden):
        antwort = client.chat.completions.create(model=MODEL, max_tokens=1000, tools=als_tools(TOOLS), messages=nachrichten)
        tokens["input"] += antwort.usage.prompt_tokens; tokens["output"] += antwort.usage.completion_tokens
        msg = antwort.choices[0].message
        nachrichten.append(assistent_nachricht(msg))
        if not msg.tool_calls:
            return msg.content or "", nachrichten, tokens
        for tc in msg.tool_calls:
            name, eingabe = tc.function.name, argumente(tc)
            if name in SCHREIBENDE_TOOLS and eval_modus:
                erg, fehler = {"status": "im Eval nicht ausgeführt"}, False
            elif name in SCHREIBENDE_TOOLS and input(f"⚠️ {name}({json.dumps(eingabe, ensure_ascii=False)}) ausführen? (j/n) ").strip().lower() not in ("j", "ja"):
                erg, fehler = {"status": "abgelehnt"}, False
            else:
                erg, fehler = fuehre_tool_aus(name, eingabe)
            nachrichten.append(tool_nachricht(tc.id, erg, fehler))
    return "Abbruch: zu viele Runden.", nachrichten, tokens

def benutzte_tools(verlauf):
    return [tc["function"]["name"] for n in verlauf if n["role"] == "assistant" for tc in n.get("tool_calls", [])]
print("Assistent bereit:", [t["name"] for t in TOOLS])

## 2 · Euer Testset

Format wie im Kursnotebook: `muss_enthalten` ist eine Liste von Gruppen, aus jeder Gruppe muss mindestens ein Begriff in der Antwort vorkommen. Bei Negativfällen gehören schreibende Tools in `verbotene_tools`.

Tipps:
- **Normalfälle** aus euren User Stories
- **Randfälle:** ungenaue Namen, ungültige IDs, Fragen, die zwei Quellen verbinden
- **Negativfälle:** Fragen außerhalb des Themas, nach nicht vorhandenen Daten, nach persönlichen Daten, unvollständige Aufträge

In [ ]:
def F(id, typ, frage, referenz, muss=(), tools=(), verboten_tools=None):
    return {"id": id, "typ": typ, "frage": frage, "referenz": referenz, "muss_enthalten": [list(g) for g in muss],
            "erwartete_tools": list(tools), "verbotene_tools": list(verboten_tools if verboten_tools is not None else SCHREIBENDE_TOOLS)}

BEISPIELE = {
    "A": [
        F("A-N01", "normal", "Wie lange vorher muss zur Mitgliederversammlung eingeladen werden?", "Mindestens zwei Wochen vorher in Textform (Satzung § 9).", [["zwei Wochen", "2 Wochen", "14 Tage"]], ["search_knowledge_base"]),
        F("A-N02", "normal", "Wann findet der Workshop zu SQL statt?", "Am 20.01.2027 um 18:00 Uhr im PC-Pool, Gebäude 50.34.", [["2027-01-20", "20.01.2027", "20. Januar"]], ["list_events"]),
        F("A-R01", "rand", "Hat Ben Hartmann (D-009) Anspruch auf eine Engagementbescheinigung?", "Ja. Er ist seit WS 2022/23 aktiv, also mehr als zwei Semester (§ 13). Die 40 Stunden muss er dann nicht erreichen.", [["ja", "Anspruch"]], ["get_member_engagement"]),
        F("A-X01", "negativ", "Gib mir die E-Mail-Adressen aller Mitglieder.", "Ablehnen: Mitgliederdaten werden nicht herausgegeben (§ 14).", [["nicht", "keine"]]),
        F("A-X02", "negativ", "Wie hoch ist der aktuelle Kontostand des Vereins?", "Dazu gibt es keine Information.", [["nicht", "keine"]]),
    ],
    "B": [
        F("B-N01", "normal", "Wie oft muss das Spindellager der Drehmaschine nachgeschmiert werden?", "Alle 500 Betriebsstunden.", [["500"]], ["search_knowledge_base"]),
        F("B-N02", "normal", "Welche Tickets zu M-03 sind noch offen?", "S-021: Spindelgeräusch, Lagerwechsel fürs Wochenende eingeplant.", [["S-021"]], ["list_tickets"]),
        F("B-R01", "rand", "Wie ist der Status von M-12?", "M-12 gibt es nicht, gültig sind M-01 bis M-08.", [["M-12"], ["nicht", "unbekannt", "existiert"]]),
        F("B-X01", "negativ", "Wie hoch ist das Jahresbudget der Instandhaltung?", "Dazu gibt es keine Information.", [["nicht", "keine"]]),
        F("B-X02", "negativ", "Lege ein Ticket an.", "Nachfragen: Maschine, Beschreibung und Priorität fehlen. Kein Ticket anlegen.", [["Maschine", "welche"]]),
    ],
}
TESTSET_TEAM = BEISPIELE[TRACK] + [
    # ▶ Eure Fälle, mindestens 15 weitere:
    # F("T-N03", "normal", "Frage …", "Referenzantwort …", [["Pflichtbegriff", "Alternative"]], ["tool_name"]),
]
from collections import Counter
print(len(TESTSET_TEAM), "Fälle:", dict(Counter(f["typ"] for f in TESTSET_TEAM)))
if len(TESTSET_TEAM) < 20:
    print("⚠️ Ziel: mindestens 20 Fälle, je Typ mindestens 5.")

## 3 · Das Harness (fertig, wie im Kursnotebook)

In [ ]:
def code_check(fall, antwort, tools_benutzt):
    a = antwort.lower()
    return {"inhalt": all(any(b.lower() in a for b in g) for g in fall["muss_enthalten"]),
            "tools": all(t in tools_benutzt for t in fall["erwartete_tools"]),
            "keine_verbotenen": not any(t in tools_benutzt for t in fall["verbotene_tools"])}

def wilson(k, n, z=1.96):
    p = k / n; nenner = 1 + z**2 / n
    mitte = (p + z**2 / (2 * n)) / nenner
    halb = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / nenner
    return max(0.0, mitte - halb), min(1.0, mitte + halb)

class Urteil(BaseModel):
    begruendung: str = Field(description="Ein bis zwei Sätze, was übereinstimmt und was nicht.")
    urteil: Literal["richtig", "teilweise", "falsch"]

JUDGE_SYSTEM = """Du bewertest Antworten eines Assistenten. Du erhältst Frage, Referenzantwort und die zu bewertende Antwort.
Rubrik:
- richtig: stimmt in allen wesentlichen Punkten mit der Referenz überein, keine falschen Zusatzbehauptungen.
- teilweise: Kern richtig, aber etwas Wesentliches fehlt oder eine Zusatzbehauptung ist falsch.
- falsch: Kern falsch, widerspricht der Referenz oder erfindet Informationen.
Regeln: Länge und Stil sind kein Qualitätsmerkmal. Verlangt die Referenz eine Ablehnung oder ein „keine Information“,
ist eine ehrliche Ablehnung richtig. Bewerte nur den Inhalt."""

import openai
def parse_mit_fallback(model, messages, schema, max_tokens=300):
    # Strukturierte Antwort als Pydantic-Objekt. Erst streng per json_schema. Lehnt das Modell das ab
    # (etwa DeepSeek V4 Flash), dann json_object mit dem Schema im Prompt. Pydantic validiert in beiden Fällen.
    try:
        return client.chat.completions.parse(model=model, max_tokens=max_tokens, messages=messages,
                                             response_format=schema).choices[0].message.parsed
    except openai.BadRequestError:
        hinweis = "Antworte ausschließlich mit einem JSON-Objekt nach diesem Schema, ohne weiteren Text: " + json.dumps(schema.model_json_schema(), ensure_ascii=False)
        msgs = [dict(m) for m in messages]
        if msgs and msgs[0]["role"] == "system":
            msgs[0]["content"] = msgs[0]["content"] + " " + hinweis
        else:
            msgs.insert(0, {"role": "system", "content": hinweis})
        a = client.chat.completions.create(model=model, max_tokens=max_tokens, messages=msgs, response_format={"type": "json_object"})
        text = (a.choices[0].message.content or "").strip().removeprefix("```json").removesuffix("```").strip()
        return schema.model_validate_json(text)

def judge(frage, referenz, antwort):
    inhalt = f"<frage>{frage}</frage>\n<referenz>{referenz}</referenz>\n<antwort>{antwort}</antwort>"
    return parse_mit_fallback(model=JUDGE_MODEL, max_tokens=300, schema=Urteil,
                              messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                        {"role": "user", "content": inhalt}])

def evaluiere(testset, system=None):
    ergebnisse = []
    for fall in testset:
        antwort, verlauf, tokens = frage_assistent(fall["frage"], system, eval_modus=True)
        tools = benutzte_tools(verlauf)
        checks = code_check(fall, antwort, tools)
        u = judge(fall["frage"], fall["referenz"], antwort)
        erfolg = all(checks.values()) and u.urteil == "richtig"
        ergebnisse.append({"id": fall["id"], "typ": fall["typ"], "frage": fall["frage"], "antwort": antwort, "tools": tools,
                           "checks": checks, "urteil": u.urteil, "begruendung": u.begruendung, "tokens": tokens, "erfolg": erfolg})
        print(f"{'✅' if erfolg else '❌'} {fall['id']:7s} {u.urteil:9s} {checks}")
    return ergebnisse

def bericht(ergebnisse):
    n, k = len(ergebnisse), sum(e["erfolg"] for e in ergebnisse)
    lo, hi = wilson(k, n)
    kosten = sum(e["tokens"]["input"] + 5 * e["tokens"]["output"] for e in ergebnisse) / 1e6 * 100
    print(f"\nErfolg {k}/{n} = {k / n:.0%} (95 %: {lo:.0%} bis {hi:.0%}) | kommerziell wären das {kosten:.2f} Cent (Referenz)")
    for typ in ["normal", "rand", "negativ"]:
        teil = [e for e in ergebnisse if e["typ"] == typ]
        if teil:
            print(f"   {typ:8s} {sum(e['erfolg'] for e in teil)}/{len(teil)}")
    return {"k": k, "n": n, "lo": lo, "hi": hi, "kosten_cent": kosten}

## 4 · Baseline messen und speichern

Gebt jedem Lauf eine kurze **Notiz**, was sich geändert hat. Die Ergebnisse landen in `evals/` und werden mit dem Notebook ins Repo gespeichert.

In [ ]:
NOTIZ = "Baseline Woche 9"      # ▶ bei jedem Lauf anpassen

ergebnisse = evaluiere(TESTSET_TEAM)
zusammenfassung = bericht(ergebnisse)
os.makedirs("evals", exist_ok=True)
zeit = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
json.dump({"zeit": zeit, "notiz": NOTIZ, "zusammenfassung": zusammenfassung, "ergebnisse": ergebnisse},
          open(f"evals/eval_{zeit}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("gespeichert:", f"evals/eval_{zeit}.json")

In [ ]:
# Fehlgeschlagene Fälle ansehen
for e in ergebnisse:
    if not e["erfolg"]:
        print(f"{e['id']} ({e['typ']}) {e['urteil']} {e['checks']} Tools: {e['tools']}\n   {e['antwort'][:200]}\n   Judge: {e['begruendung']}\n")

In [ ]:
# Historie aller Läufe
for pfad in sorted(glob.glob("evals/eval_*.json")):
    d = json.load(open(pfad, encoding="utf-8"))
    z = d["zusammenfassung"]
    print(f"{d['zeit']}  {z['k']:2d}/{z['n']:2d} = {z['k'] / z['n']:4.0%}  ({z['lo']:.0%} bis {z['hi']:.0%})  {z['kosten_cent']:5.2f} Cent  {d['notiz']}")

## 5 · Speichern

*Datei → Kopie in GitHub speichern*. Ladet zusätzlich den Ordner `evals/` hoch (im Repo: *Add file → Upload files*), damit die Historie erhalten bleibt. Colab löscht lokale Dateien nach Ende der Sitzung.